<a href="https://colab.research.google.com/github/ShahdKhatib/ArabicNumeralRecognition/blob/main/Classic_and_Deep_Learning_Handwritten_Arabic_Numerals_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Handwritten Arabic Numerals Recognition (0–9)**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Vision_Numbers"

DATASET_PATH = os.path.join(PROJECT_PATH, "dataset")
MODELS_PATH = os.path.join(PROJECT_PATH, "models")
RESULTS_PATH = os.path.join(PROJECT_PATH, "results")
CACHE_PATH = os.path.join(PROJECT_PATH, "cache")

# create folders safely
os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(CACHE_PATH, exist_ok=True)

print("Project structure:")
print(os.listdir(PROJECT_PATH))


# **Preprocessing**

In [ ]:
# =========================
# 4. Load dataset with better preprocessing and cache
# =========================

import os
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

IMG_SIZE = 28
PREPROCESS_VERSION = "v3_aspect_pad"

os.makedirs(CACHE_PATH, exist_ok=True)
processed_data_file = os.path.join(CACHE_PATH, f"processed_dataset_{PREPROCESS_VERSION}.npz")


def preprocess_digit_image(image_path, output_size=28, threshold=245, inner_size=22):
    """
    Preprocess digit image by:
    - loading original size
    - converting to grayscale
    - finding foreground region
    - cropping to digit content
    - resizing with aspect ratio preserved
    - centering on white 28x28 canvas
    """
    img = Image.open(image_path).convert("L")
    img_array = np.array(img, dtype=np.uint8)

    # foreground = darker than near-white background
    mask = img_array < threshold

    # if foreground not found, just resize carefully with aspect ratio
    if not np.any(mask):
        img.thumbnail((inner_size, inner_size), Image.Resampling.LANCZOS)
        canvas = Image.new("L", (output_size, output_size), 255)
        x = (output_size - img.size[0]) // 2
        y = (output_size - img.size[1]) // 2
        canvas.paste(img, (x, y))
        return np.array(canvas, dtype=np.float32)

    coords = np.argwhere(mask)
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0) + 1

    # crop to foreground
    cropped = img_array[y_min:y_max, x_min:x_max]
    cropped_img = Image.fromarray(cropped)

    # preserve aspect ratio while resizing
    cropped_img.thumbnail((inner_size, inner_size), Image.Resampling.LANCZOS)

    # paste centered on white canvas
    canvas = Image.new("L", (output_size, output_size), 255)
    x = (output_size - cropped_img.size[0]) // 2
    y = (output_size - cropped_img.size[1]) // 2
    canvas.paste(cropped_img, (x, y))

    return np.array(canvas, dtype=np.float32)


if os.path.exists(processed_data_file):
    print("Loading preprocessed dataset from cache...")
    data = np.load(processed_data_file)
    X = data["X"]
    y = data["y"]
    print("Loaded cached dataset successfully.")
else:
    print("No cached dataset found. Loading and preprocessing images from folders...")

    images = []
    labels = []

    class_folders = sorted(
        [d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))],
        key=lambda x: int(x)
    )

    for class_name in tqdm(class_folders, desc="Processing class folders"):
        class_path = os.path.join(DATASET_PATH, class_name)
        file_list = sorted(os.listdir(class_path))

        for file_name in tqdm(file_list, desc=f"Class {class_name}", leave=False):
            file_path = os.path.join(class_path, file_name)

            try:
                img_array = preprocess_digit_image(
                    file_path,
                    output_size=IMG_SIZE,
                    threshold=245,
                    inner_size=22
                )
                images.append(img_array)
                labels.append(int(class_name))

            except Exception as e:
                print(f"Error loading {file_path}: {e}")

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)


    os.makedirs(CACHE_PATH, exist_ok=True)
    np.savez_compressed(processed_data_file, X=X, y=y)
    print(f"Saved processed dataset to cache: {processed_data_file}")

print("Loaded images shape:", X.shape)
print("Loaded labels shape:", y.shape)
print("Unique labels:", np.unique(y))
print("Pixel range:", X.min(), "to", X.max())

In [ ]:
# =========================
# 5. Visualize sample images
# =========================

import matplotlib.pyplot as plt
import random

print("Displaying random sample images...")

plt.figure(figsize=(12, 6))
sample_indices = random.sample(range(len(X)), 20)

for i, idx in enumerate(sample_indices):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X[idx], cmap="gray")
    plt.title(f"Label: {y[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import os
import random

sample_class = random.choice(os.listdir(DATASET_PATH))
sample_file = random.choice(os.listdir(os.path.join(DATASET_PATH, sample_class)))

img = Image.open(os.path.join(DATASET_PATH, sample_class, sample_file))

print("Original image size:", img.size)

In [ ]:
# =========================
# 6. Inspect pixel range
# =========================

print("Inspecting dataset properties...")
print("X dtype:", X.dtype)
print("Min pixel value:", X.min())
print("Max pixel value:", X.max())
print("Single image shape:", X[0].shape)

In [ ]:
# =========================
# 7. Normalize and reshape, then cache
# =========================

cnn_ready_file = os.path.join(CACHE_PATH, "cnn_ready_dataset.npz")

if os.path.exists(cnn_ready_file):
    print("Loading normalized CNN-ready data from cache...")
    data = np.load(cnn_ready_file)
    X_cnn = data["X_cnn"]
    y = data["y"]
    print("Loaded CNN-ready dataset successfully.")
else:
    print("Normalizing and reshaping data for CNN...")
    X_normalized = X / 255.0
    X_cnn = X_normalized.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

    np.savez_compressed(cnn_ready_file, X_cnn=X_cnn, y=y)
    print(f"Saved CNN-ready dataset to cache: {cnn_ready_file}")

print("CNN input shape:", X_cnn.shape)
print("CNN input range:", X_cnn.min(), "to", X_cnn.max())

In [ ]:
# =========================
# 8. Train / validation / test split with caching
# =========================

from sklearn.model_selection import train_test_split

split_data_file = os.path.join(CACHE_PATH, "data_splits.npz")

if os.path.exists(split_data_file):
    print("Loading saved train/validation/test splits from cache...")
    data = np.load(split_data_file)

    X_train = data["X_train"]
    X_val = data["X_val"]
    X_test = data["X_test"]
    y_train = data["y_train"]
    y_val = data["y_val"]
    y_test = data["y_test"]

    print("Loaded saved splits successfully.")
else:
    print("Creating train/validation/test splits...")

    X_train, X_temp, y_train, y_temp = train_test_split(
        X_cnn, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    np.savez_compressed(
        split_data_file,
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test
    )

    print(f"Saved splits to cache: {split_data_file}")

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

In [ ]:
# =========================
# 9. Check class distribution after split
# =========================

from collections import Counter

print("Checking class distribution...")
print("Train class distribution:", Counter(y_train))
print("Validation class distribution:", Counter(y_val))
print("Test class distribution:", Counter(y_test))

In [ ]:
# =========================
# 10. Save preprocessing summary
# =========================

summary_file = os.path.join(CACHE_PATH, "preprocessing_summary.txt")

with open(summary_file, "w") as f:
    f.write("Preprocessing Summary\n")
    f.write("=====================\n")
    f.write(f"Original dataset path: {DATASET_PATH}\n")
    f.write(f"Image size used: {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"Total samples: {len(y)}\n")
    f.write(f"X shape: {X.shape}\n")
    f.write(f"X_cnn shape: {X_cnn.shape}\n")
    f.write(f"Train shape: {X_train.shape}, {y_train.shape}\n")
    f.write(f"Validation shape: {X_val.shape}, {y_val.shape}\n")
    f.write(f"Test shape: {X_test.shape}, {y_test.shape}\n")

print(f"Saved preprocessing summary to: {summary_file}")

# **Build the baseline model (HOG + SVM)**

In [ ]:
cnn_ready_file = os.path.join(CACHE_PATH, "cnn_ready_dataset_v3_aspect_pad.npz")
split_data_file = os.path.join(CACHE_PATH, "data_splits_v3_aspect_pad.npz")
summary_file = os.path.join(CACHE_PATH, "preprocessing_summary_v3_aspect_pad.txt")
hog_features_file = os.path.join(CACHE_PATH, "hog_features_v3_aspect_pad.npz")
baseline_results_file = os.path.join(RESULTS_PATH, "baseline_hog_svm_results_v3_aspect_pad.txt")

# preprocess for HOG

In [ ]:
# =========================
# 11. Prepare grayscale splits for HOG
# =========================

print("Preparing grayscale data for HOG...")

X_train_gray = X_train.squeeze(-1)
X_val_gray = X_val.squeeze(-1)
X_test_gray = X_test.squeeze(-1)

print("X_train_gray shape:", X_train_gray.shape)
print("X_val_gray shape:", X_val_gray.shape)
print("X_test_gray shape:", X_test_gray.shape)

In [ ]:
# =========================
# 12. Extract HOG features with caching
# =========================

import os
import numpy as np
from tqdm.auto import tqdm
from skimage.feature import hog

hog_features_file = os.path.join(CACHE_PATH, "hog_features_v3_aspect_pad.npz")

def extract_hog_features(images, desc="Extracting HOG"):
    features = []
    for img in tqdm(images, desc=desc):
        feat = hog(
            img,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )
        features.append(feat)
    return np.array(features, dtype=np.float32)

if os.path.exists(hog_features_file):
    print("Loading HOG features from cache...")
    data = np.load(hog_features_file)

    X_train_hog = data["X_train_hog"]
    X_val_hog = data["X_val_hog"]
    X_test_hog = data["X_test_hog"]

    print("Loaded cached HOG features successfully.")
else:
    print("No cached HOG features found. Extracting now...")

    X_train_hog = extract_hog_features(X_train_gray, desc="HOG train")
    X_val_hog = extract_hog_features(X_val_gray, desc="HOG val")
    X_test_hog = extract_hog_features(X_test_gray, desc="HOG test")

    np.savez_compressed(
        hog_features_file,
        X_train_hog=X_train_hog,
        X_val_hog=X_val_hog,
        X_test_hog=X_test_hog
    )

    print(f"Saved HOG features to cache: {hog_features_file}")

print("X_train_hog shape:", X_train_hog.shape)
print("X_val_hog shape:", X_val_hog.shape)
print("X_test_hog shape:", X_test_hog.shape)

In [ ]:
# =========================
# 13. Train HOG + SVM baseline (with progress indicator)
# =========================

from sklearn.svm import SVC
from tqdm.auto import tqdm
import time

print("Training HOG + SVM baseline...")

print(f"Training samples: {X_train_hog.shape[0]}")
print(f"Feature dimension: {X_train_hog.shape[1]}")

svm_clf = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    random_state=42
)

start_time = time.time()

# fake progress bar to indicate activity during training
with tqdm(total=1, desc="SVM training", bar_format="{l_bar}{bar}| elapsed: {elapsed}") as pbar:
    svm_clf.fit(X_train_hog, y_train)
    pbar.update(1)

end_time = time.time()

print("Baseline SVM training complete.")
print(f"Training time: {end_time - start_time:.2f} seconds")

In [ ]:
# =========================
# 14. Evaluate HOG + SVM baseline
# =========================

from sklearn.metrics import accuracy_score, classification_report

print("Evaluating baseline model...")

# Predictions
y_val_pred = svm_clf.predict(X_val_hog)
y_test_pred = svm_clf.predict(X_test_hog)

# Accuracy
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred, digits=4))

In [ ]:
# =========================
# 15. Confusion matrix for baseline
# =========================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

print("Plotting confusion matrix...")

cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Baseline HOG + SVM Confusion Matrix")
plt.show()

In [ ]:
# =========================
# 16. Save baseline results
# =========================

import os
from sklearn.metrics import classification_report

baseline_results_file = os.path.join(RESULTS_PATH, "baseline_hog_svm_results_v3_aspect_pad.txt")

with open(baseline_results_file, "w") as f:
    f.write("Baseline Model: HOG + SVM\n")
    f.write("=========================\n\n")

    f.write("Accuracy\n")
    f.write("--------\n")
    f.write(f"Validation Accuracy: {val_acc:.6f}\n")
    f.write(f"Test Accuracy: {test_acc:.6f}\n\n")

    f.write("Classification Report (Test)\n")
    f.write("----------------------------\n")
    f.write(classification_report(y_test, y_test_pred, digits=6))

print(f"Saved baseline results to: {baseline_results_file}")

# **CNN MODEL (DEEP LEARNING)**

In [ ]:
cnn_ready_file = os.path.join(CACHE_PATH, "cnn_ready_dataset_v3_aspect_pad.npz")
split_data_file = os.path.join(CACHE_PATH, "data_splits_v3_aspect_pad.npz")
cnn_model_file = os.path.join(MODELS_PATH, "best_cnn_v3_aspect_pad.keras")
cnn_results_file = os.path.join(RESULTS_PATH, "cnn_results_v3_aspect_pad.txt")

In [ ]:
# =========================
# 17. Data augmentation for CNN
# =========================

import tensorflow as tf
from tensorflow.keras import layers

print("Creating data augmentation pipeline...")

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.03),
    layers.RandomTranslation(height_factor=0.03, width_factor=0.03)
], name="data_augmentation")

print("Data augmentation pipeline ready.")

In [ ]:
# =========================
# 18. Build CNN model
# =========================

from tensorflow.keras import models, layers

print("Building CNN model...")

cnn_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    data_augmentation,

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.15),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.20),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.25),

    layers.Dense(10, activation='softmax')
])

cnn_model.summary()

In [ ]:
# =========================
# 19. Compile CNN model
# =========================

print("Compiling CNN model...")

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("CNN model compiled successfully.")

In [ ]:
# =========================
# 20. Define callbacks
# =========================

from tensorflow.keras import callbacks
import os

cnn_model_file = os.path.join(MODELS_PATH, "best_cnn_v3_aspect_pad.keras")

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

model_checkpoint = callbacks.ModelCheckpoint(
    filepath=cnn_model_file,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

callback_list = [early_stopping, reduce_lr, model_checkpoint]

print("Callbacks ready.")
print("Best model will be saved to:", cnn_model_file)

In [ ]:
from tensorflow.keras.models import load_model

print("Loading best saved CNN model...")

cnn_model = load_model("/content/drive/MyDrive/Vision_Numbers/models/best_cnn_v3_aspect_pad.keras")

print("Model loaded successfully.")

In [ ]:
# =========================
# 21. Train CNN model (only if needed)
# =========================

import os
import time
from tensorflow.keras.models import load_model

cnn_model_file = os.path.join(MODELS_PATH, "best_cnn_v3_aspect_pad.keras")

if os.path.exists(cnn_model_file):
    print("Existing trained CNN model found.")
    print("Loading saved model instead of retraining...")

    cnn_model = load_model(cnn_model_file)

    history = None

else:
    print("No saved CNN model found. Starting training...")

    print("Train shape:", X_train.shape, y_train.shape)
    print("Validation shape:", X_val.shape, y_val.shape)

    start_time = time.time()

    history = cnn_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        callbacks=callback_list,
        verbose=1
    )

    end_time = time.time()

    print("CNN training complete.")
    print(f"Training time: {end_time - start_time:.2f} seconds")

In [ ]:
# =========================
# 23. Evaluate CNN model
# =========================

from sklearn.metrics import accuracy_score, classification_report

print("Evaluating CNN model...")

# Predict probabilities
val_probs = cnn_model.predict(X_val, verbose=1)
test_probs = cnn_model.predict(X_test, verbose=1)

# Top-1 predictions
y_val_pred_cnn = np.argmax(val_probs, axis=1)
y_test_pred_cnn = np.argmax(test_probs, axis=1)

# Accuracy
cnn_val_acc = accuracy_score(y_val, y_val_pred_cnn)
cnn_test_acc = accuracy_score(y_test, y_test_pred_cnn)

print(f"Validation Accuracy: {cnn_val_acc:.4f}")
print(f"Test Accuracy: {cnn_test_acc:.4f}")

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred_cnn, digits=4))

In [ ]:
# =========================
# 24. Confusion matrix for CNN
# =========================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

print("Plotting CNN confusion matrix...")

cnn_cm = confusion_matrix(y_test, y_test_pred_cnn)

fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cnn_cm, display_labels=list(range(10)))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("CNN Confusion Matrix")
plt.show()

In [ ]:
# =========================
# 25. Show misclassified test examples
# =========================

import matplotlib.pyplot as plt
import numpy as np

print("Displaying misclassified test examples...")

misclassified_indices = np.where(y_test != y_test_pred_cnn)[0]

print(f"Number of misclassified test samples: {len(misclassified_indices)}")

num_to_show = min(16, len(misclassified_indices))

selected_indices = misclassified_indices[:num_to_show]

plt.figure(figsize=(10, 10))

for i, idx in enumerate(selected_indices):
    plt.subplot(4, 4, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    plt.title(f"True: {y_test[idx]} | Pred: {y_test_pred_cnn[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 25. Save CNN results
# =========================

import os
from sklearn.metrics import classification_report

cnn_results_file = os.path.join(RESULTS_PATH, "cnn_results_v3_aspect_pad.txt")

with open(cnn_results_file, "w") as f:
    f.write("CNN Model Results\n")
    f.write("=================\n\n")

    f.write("Accuracy\n")
    f.write("--------\n")
    f.write(f"Validation Accuracy: {cnn_val_acc:.6f}\n")
    f.write(f"Test Accuracy: {cnn_test_acc:.6f}\n\n")

    f.write("Classification Report (Test)\n")
    f.write("----------------------------\n")
    f.write(classification_report(y_test, y_test_pred_cnn, digits=6))

print(f"Saved CNN results to: {cnn_results_file}")

In [ ]:
# =========================
# 26. Compare baseline vs CNN
# =========================

import pandas as pd

comparison_df = pd.DataFrame({
    "Model": ["HOG + SVM Baseline", "CNN"],
    "Validation Accuracy": [0.8131, cnn_val_acc],
    "Test Accuracy": [0.8190, cnn_test_acc],
})

print("Baseline vs CNN comparison:")
display(comparison_df.style.format({
    "Validation Accuracy": "{:.4f}",
    "Test Accuracy": "{:.4f}",
}))

In [ ]:
# =========================
# FINAL SUMMARY CELL FOR REPORT METRICS
# =========================

import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Collecting final report metrics...")

# -------------------------------------------------
# 1. Recompute / confirm baseline metrics
# -------------------------------------------------
y_val_pred_baseline = svm_clf.predict(X_val_hog)
y_test_pred_baseline = svm_clf.predict(X_test_hog)

baseline_val_acc = accuracy_score(y_val, y_val_pred_baseline)
baseline_test_acc = accuracy_score(y_test, y_test_pred_baseline)

baseline_report_dict = classification_report(
    y_test, y_test_pred_baseline, output_dict=True, digits=6
)
baseline_report_df = pd.DataFrame(baseline_report_dict).transpose()

baseline_cm = confusion_matrix(y_test, y_test_pred_baseline)

# -------------------------------------------------
# 2. Recompute / confirm CNN metrics
# -------------------------------------------------
val_probs = cnn_model.predict(X_val, verbose=0)
test_probs = cnn_model.predict(X_test, verbose=0)

y_val_pred_cnn = np.argmax(val_probs, axis=1)
y_test_pred_cnn = np.argmax(test_probs, axis=1)

cnn_val_acc = accuracy_score(y_val, y_val_pred_cnn)
cnn_test_acc = accuracy_score(y_test, y_test_pred_cnn)

cnn_report_dict = classification_report(
    y_test, y_test_pred_cnn, output_dict=True, digits=6
)
cnn_report_df = pd.DataFrame(cnn_report_dict).transpose()

cnn_cm = confusion_matrix(y_test, y_test_pred_cnn)

cnn_misclassified_indices = np.where(y_test != y_test_pred_cnn)[0]
cnn_num_misclassified = len(cnn_misclassified_indices)

# -------------------------------------------------
# 3. Training history summary (if available)
# -------------------------------------------------
history_summary = {}

if "history" in globals() and history is not None:
    history_dict = history.history

    history_summary = {
        "best_train_accuracy": float(np.max(history_dict["accuracy"])),
        "best_val_accuracy": float(np.max(history_dict["val_accuracy"])),
        "best_train_loss": float(np.min(history_dict["loss"])),
        "best_val_loss": float(np.min(history_dict["val_loss"])),
        "epochs_ran": len(history_dict["accuracy"]),
        "best_val_acc_epoch": int(np.argmax(history_dict["val_accuracy"]) + 1),
        "best_val_loss_epoch": int(np.argmin(history_dict["val_loss"]) + 1),
    }
else:
    history_summary = {
        "best_train_accuracy": None,
        "best_val_accuracy": None,
        "best_train_loss": None,
        "best_val_loss": None,
        "epochs_ran": None,
        "best_val_acc_epoch": None,
        "best_val_loss_epoch": None,
    }

# -------------------------------------------------
# 4. Main comparison table
# -------------------------------------------------
comparison_df = pd.DataFrame({
    "Model": ["HOG + SVM", "CNN"],
    "Validation Accuracy": [baseline_val_acc, cnn_val_acc],
    "Test Accuracy": [baseline_test_acc, cnn_test_acc],
})

# -------------------------------------------------
# 5. Compact per-class table for CNN only
#    (usually the most important one for report)
# -------------------------------------------------
cnn_per_class_df = cnn_report_df.loc[
    [str(i) for i in range(10)],
    ["precision", "recall", "f1-score", "support"]
].copy()
cnn_per_class_df.index.name = "Class"

# -------------------------------------------------
# 6. Compact per-class table for baseline
# -------------------------------------------------
baseline_per_class_df = baseline_report_df.loc[
    [str(i) for i in range(10)],
    ["precision", "recall", "f1-score", "support"]
].copy()
baseline_per_class_df.index.name = "Class"

# -------------------------------------------------
# 7. Print everything cleanly
# -------------------------------------------------
print("\n=== MAIN MODEL COMPARISON ===")
display(comparison_df.style.format({
    "Validation Accuracy": "{:.4f}",
    "Test Accuracy": "{:.4f}",
}))

print("\n=== CNN PER-CLASS METRICS (TEST) ===")
display(cnn_per_class_df.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1-score": "{:.4f}",
    "support": "{:.0f}",
}))

print("\n=== BASELINE PER-CLASS METRICS (TEST) ===")
display(baseline_per_class_df.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1-score": "{:.4f}",
    "support": "{:.0f}",
}))

print("\n=== CNN CONFUSION MATRIX ===")
print(cnn_cm)

print("\n=== BASELINE CONFUSION MATRIX ===")
print(baseline_cm)

print("\n=== CNN MISCLASSIFICATION SUMMARY ===")
print(f"Misclassified test samples: {cnn_num_misclassified} / {len(y_test)}")
print(f"Misclassification rate: {cnn_num_misclassified / len(y_test):.4f}")

print("\n=== TRAINING HISTORY SUMMARY ===")
for k, v in history_summary.items():
    print(f"{k}: {v}")

# -------------------------------------------------
# 8. Save all report-ready metrics to files
# -------------------------------------------------
summary_txt_file = os.path.join(RESULTS_PATH, "final_report_metrics_summary.txt")
comparison_csv_file = os.path.join(RESULTS_PATH, "model_comparison_table.csv")
cnn_per_class_csv_file = os.path.join(RESULTS_PATH, "cnn_per_class_metrics.csv")
baseline_per_class_csv_file = os.path.join(RESULTS_PATH, "baseline_per_class_metrics.csv")
cnn_cm_csv_file = os.path.join(RESULTS_PATH, "cnn_confusion_matrix.csv")
baseline_cm_csv_file = os.path.join(RESULTS_PATH, "baseline_confusion_matrix.csv")
history_json_file = os.path.join(RESULTS_PATH, "cnn_training_history_summary.json")

comparison_df.to_csv(comparison_csv_file, index=False)
cnn_per_class_df.to_csv(cnn_per_class_csv_file)
baseline_per_class_df.to_csv(baseline_per_class_csv_file)
pd.DataFrame(cnn_cm).to_csv(cnn_cm_csv_file, index=False)
pd.DataFrame(baseline_cm).to_csv(baseline_cm_csv_file, index=False)

with open(history_json_file, "w") as f:
    json.dump(history_summary, f, indent=4)

with open(summary_txt_file, "w") as f:
    f.write("FINAL REPORT METRICS SUMMARY\n")
    f.write("============================\n\n")

    f.write("MAIN MODEL COMPARISON\n")
    f.write("---------------------\n")
    f.write(comparison_df.to_string(index=False))
    f.write("\n\n")

    f.write("CNN PER-CLASS METRICS (TEST)\n")
    f.write("----------------------------\n")
    f.write(cnn_per_class_df.to_string())
    f.write("\n\n")

    f.write("BASELINE PER-CLASS METRICS (TEST)\n")
    f.write("---------------------------------\n")
    f.write(baseline_per_class_df.to_string())
    f.write("\n\n")

    f.write("CNN CONFUSION MATRIX\n")
    f.write("--------------------\n")
    f.write(pd.DataFrame(cnn_cm).to_string(index=False))
    f.write("\n\n")

    f.write("BASELINE CONFUSION MATRIX\n")
    f.write("-------------------------\n")
    f.write(pd.DataFrame(baseline_cm).to_string(index=False))
    f.write("\n\n")

    f.write("CNN MISCLASSIFICATION SUMMARY\n")
    f.write("-----------------------------\n")
    f.write(f"Misclassified test samples: {cnn_num_misclassified} / {len(y_test)}\n")
    f.write(f"Misclassification rate: {cnn_num_misclassified / len(y_test):.6f}\n\n")

    f.write("TRAINING HISTORY SUMMARY\n")
    f.write("------------------------\n")
    for k, v in history_summary.items():
        f.write(f"{k}: {v}\n")

print("\nSaved report-ready files:")
print(summary_txt_file)
print(comparison_csv_file)
print(cnn_per_class_csv_file)
print(baseline_per_class_csv_file)
print(cnn_cm_csv_file)
print(baseline_cm_csv_file)
print(history_json_file)